# Practice P5: answers

Try each problem for real before you read its answer. There's more than one right way; these are one way each.

In [ ]:
import sys
sys.path.append("..")  # checks.py is one folder up
from checks import check

In [ ]:
# Run this cell before problems 10 and 12.
# ListNode and TreeNode are the same as in notebook 20.

class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

def build_list(values):
    # ["a", "b"] -> a linked list a -> b
    head = None
    for v in values[::-1]:
        head = ListNode(v, head)
    return head

def to_list(head):
    # a linked list -> a normal Python list, so the checks are easy to read
    values = []
    node = head
    while node:
        values.append(node.val)
        node = node.next
    return values

## 1. Unpack the packing list

Every item is either a plain thing or a smaller list of the same kind. Plain things go straight into the answer. A list is the same problem again, so `flatten` calls itself on it and adds what comes back. There's no separate base case: a list with no lists inside never makes a recursive call.

In [ ]:
def flatten(items):
    result = []
    for item in items:
        if type(item) == list:
            for inner in flatten(item):   # same problem, smaller suitcase
                result.append(inner)
        else:
            result.append(item)
    return result

In [ ]:
check(flatten([1, [2, 3], 4]), [1, 2, 3, 4])
check(flatten(["a", ["b", ["c"]], "d"]), ["a", "b", "c", "d"])
check(flatten([1, 2, 3]), [1, 2, 3])
check(flatten([[[[5]]]]), [5])
check(flatten([]), [])
check(flatten([[], [[]]]), [])
check(flatten(["passport", ["shirt", ["socks", "socks"]], "charger"]), ["passport", "shirt", "socks", "socks", "charger"])

## 2. The most used words in your diary

Clean the words first, then count them, then sort. The sort key `(-counts[w], w)` means "biggest count first, and alphabetical when counts are equal". A slice `[:k]` handles both "take k" and "fewer than k".

In [ ]:
from collections import Counter

def top_words(text, k):
    words = []
    for raw in text.split():
        word = raw.lower().strip(".,!?")
        if word != "":
            words.append(word)
    counts = Counter(words)
    ordered = sorted(counts, key=lambda w: (-counts[w], w))
    return ordered[:k]

In [ ]:
check(top_words("The cat sat. The cat ran! A dog sat?", 2), ["cat", "sat"])
check(top_words("Tea, tea, TEA! coffee", 1), ["tea"])
check(top_words("b a b a c", 5), ["a", "b", "c"])
check(top_words("Hi! ... hi", 2), ["hi"])
check(top_words("one two two three three three", 3), ["three", "two", "one"])
check(top_words("", 3), [])
check(top_words("sun moon", 0), [])

## 3. A smooth temperature display

A `deque` holds the readings in order: new ones go in on the right, the oldest comes out on the left. Keeping `self.total` up to date means each `add` does a few steps, however many readings there are.

In [ ]:
from collections import deque

class MovingAverage:
    def __init__(self, size):
        self.size = size
        self.readings = deque()
        self.total = 0

    def add(self, value):
        self.readings.append(value)
        self.total += value
        if len(self.readings) > self.size:
            self.total -= self.readings.popleft()   # the oldest falls out
        return round(self.total / len(self.readings), 2)

In [ ]:
m = MovingAverage(3)
check(m.add(10), 10.0)
check(m.add(20), 15.0)
check(m.add(30), 20.0)
check(m.add(40), 30.0)
check(m.add(-5), 21.67)

one = MovingAverage(1)
check(one.add(7), 7.0)
check(one.add(9), 9.0)

## 4. Every lunchbox you could pack

Every lunchbox either has the first food or doesn't. So: solve the rest of the list, then make two copies of each answer, one without the first food and one with it in front. The empty list is the base case: it has exactly one lunchbox, `[]`.

In [ ]:
def all_combos(items):
    if items == []:
        return [[]]
    first = items[0]
    without_first = all_combos(items[1:])
    result = []
    for box in without_first:
        result.append(box)              # without the first food
        result.append([first] + box)    # with the first food in front
    return sorted(result)

In [ ]:
check(all_combos(["dal", "roti"]), [[], ["dal"], ["dal", "roti"], ["roti"]])
check(all_combos([]), [[]])
check(all_combos(["a"]), [[], ["a"]])
check(all_combos([1, 2, 3]), [[], [1], [1, 2], [1, 2, 3], [1, 3], [2], [2, 3], [3]])
check(all_combos([3, 1]), [[], [1], [3], [3, 1]])

## 5. All the ways to arrange the letters

Choose the first letter, then arrange the rest: the same problem with a shorter word. Repeated letters make the same word more than once, so a set removes the copies, and `sorted` gives the fixed order.

In [ ]:
def arrangements(word):
    if len(word) <= 1:
        return [word]
    found = set()
    for i in range(len(word)):
        first = word[i]
        rest = word[:i] + word[i + 1:]
        for tail in arrangements(rest):
            found.add(first + tail)
    return sorted(found)

In [ ]:
check(arrangements("abc"), ["abc", "acb", "bac", "bca", "cab", "cba"])
check(arrangements("aab"), ["aab", "aba", "baa"])
check(arrangements("a"), ["a"])
check(arrangements(""), [""])
check(arrangements("aabb"), ["aabb", "abab", "abba", "baab", "baba", "bbaa"])
check(arrangements("zzz"), ["zzz"])

## 6. A bank account that keeps a record

The account remembers its owner, balance and a list of lines. Every method decides "allowed or refused", changes the balance if allowed, and appends exactly one line. `transfer` reaches into the other object to update its balance and its statement too.

In [ ]:
class BankAccount:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self.balance = balance
        self.lines = []

    def can_take(self, amount):
        return 0 < amount <= self.balance

    def deposit(self, amount):
        if amount <= 0:
            self.lines.append(f"refused deposit {amount}")
            return False
        self.balance += amount
        self.lines.append(f"deposit {amount}, balance {self.balance}")
        return True

    def withdraw(self, amount):
        if not self.can_take(amount):
            self.lines.append(f"refused withdraw {amount}")
            return False
        self.balance -= amount
        self.lines.append(f"withdraw {amount}, balance {self.balance}")
        return True

    def transfer(self, other, amount):
        if not self.can_take(amount):
            self.lines.append(f"refused transfer {amount}")
            return False
        self.balance -= amount
        other.balance += amount
        self.lines.append(f"transfer {amount} to {other.owner}, balance {self.balance}")
        other.lines.append(f"received {amount} from {self.owner}, balance {other.balance}")
        return True

    def get_balance(self):
        return self.balance

    def statement(self):
        return self.lines

In [ ]:
a = BankAccount("Tanvi", 100)
b = BankAccount("Ravi")
check(a.deposit(50), True)
check(a.withdraw(500), False)
check(a.withdraw(-5), False)
check(a.withdraw(30), True)
check(a.transfer(b, 20), True)
check(a.transfer(b, 1000), False)
check(a.deposit(0), False)
check(a.get_balance(), 100)
check(b.get_balance(), 20)
check(a.statement(), [
    "deposit 50, balance 150",
    "refused withdraw 500",
    "refused withdraw -5",
    "withdraw 30, balance 120",
    "transfer 20 to Ravi, balance 100",
    "refused transfer 1000",
    "refused deposit 0",
])
check(b.statement(), ["received 20 from Tanvi, balance 20"])

## 7. The school library

Keep two dictionaries: copies on the shelf per title, and a set of titles per person (`defaultdict(set)`). `lend` checks the three refusal rules before changing anything. Sorting happens only when someone asks for a list.

In [ ]:
from collections import defaultdict

class Library:
    def __init__(self):
        self.shelf = {}                   # title -> copies on the shelf
        self.loans = defaultdict(set)     # person -> titles they have

    def add_book(self, title, copies=1):
        self.shelf[title] = self.shelf.get(title, 0) + copies

    def lend(self, title, person):
        if self.shelf.get(title, 0) == 0:
            return False                  # unknown title, or none left
        if title in self.loans[person]:
            return False                  # already has a copy
        self.shelf[title] -= 1
        self.loans[person].add(title)
        return True

    def give_back(self, title, person):
        if title not in self.loans[person]:
            return False
        self.loans[person].remove(title)
        self.shelf[title] += 1
        return True

    def available(self, title):
        return self.shelf.get(title, 0)

    def books_of(self, person):
        return sorted(self.loans[person])

    def who_has(self, title):
        people = []
        for person, titles in self.loans.items():
            if title in titles:
                people.append(person)
        return sorted(people)

In [ ]:
lib = Library()
lib.add_book("Matilda", 2)
lib.add_book("Dune")
lib.add_book("Matilda")
check(lib.available("Matilda"), 3)
check(lib.lend("Dune", "Tanvi"), True)
check(lib.lend("Dune", "Ravi"), False)        # no copies left
check(lib.lend("Matilda", "Tanvi"), True)
check(lib.lend("Matilda", "Tanvi"), False)    # already has one
check(lib.lend("Hobbit", "Ravi"), False)      # library doesn't have it
check(lib.lend("Matilda", "Asha"), True)
check(lib.books_of("Tanvi"), ["Dune", "Matilda"])
check(lib.who_has("Matilda"), ["Asha", "Tanvi"])
check(lib.give_back("Dune", "Ravi"), False)   # Ravi never had it
check(lib.give_back("Dune", "Tanvi"), True)
check(lib.available("Dune"), 1)
check(lib.available("Hobbit"), 0)
check(lib.books_of("Nobody"), [])

## 8. A text editor with undo

A stack of "how the text looked before" makes undo easy: push before each change, pop to go back. `"abc"[:-0]` is `""` (because `-0` is `0`), so `delete` slices by how many characters to keep instead.

In [ ]:
class TextEditor:
    def __init__(self):
        self.text = ""
        self.before = []          # stack of older texts

    def write(self, text):
        self.before.append(self.text)
        self.text = self.text + text

    def delete(self, n):
        self.before.append(self.text)
        keep = max(len(self.text) - n, 0)
        self.text = self.text[:keep]

    def undo(self):
        if self.before:
            self.text = self.before.pop()

    def get_text(self):
        return self.text

In [ ]:
e = TextEditor()
e.write("Hello")
e.write(" world")
check(e.get_text(), "Hello world")
e.delete(6)
check(e.get_text(), "Hello")
e.undo()
check(e.get_text(), "Hello world")
e.undo()
check(e.get_text(), "Hello")
e.undo()
check(e.get_text(), "")
e.undo()                       # nothing left to undo
check(e.get_text(), "")
e.write("abc")
e.delete(0)
check(e.get_text(), "abc")
e.delete(10)
check(e.get_text(), "")
e.undo()
check(e.get_text(), "abc")

## 9. Who does the doctor see next?

A heap always gives the smallest tuple first. Putting an arrival number second in the tuple makes "same urgency" fall back to "who came first", and the name is never compared.

In [ ]:
import heapq

class ClinicQueue:
    def __init__(self):
        self.heap = []
        self.arrivals = 0

    def arrive(self, name, urgency):
        self.arrivals += 1
        heapq.heappush(self.heap, (urgency, self.arrivals, name))

    def next_patient(self):
        if not self.heap:
            return None
        urgency, order, name = heapq.heappop(self.heap)
        return name

    def waiting(self):
        return len(self.heap)

In [ ]:
q = ClinicQueue()
check(q.next_patient(), None)
q.arrive("Riya", 3)
q.arrive("Aman", 1)
q.arrive("Kabir", 3)
q.arrive("Meera", 1)
check(q.waiting(), 4)
check(q.next_patient(), "Aman")
check(q.next_patient(), "Meera")
check(q.next_patient(), "Riya")
q.arrive("Zoya", 2)
check(q.next_patient(), "Zoya")
check(q.waiting(), 1)
check(q.next_patient(), "Kabir")
check(q.next_patient(), None)
check(q.waiting(), 0)

## 10. Cancelled stations

The head is special, because nothing points to it, so skip cancelled stations at the front first. After that, stand on a station that stays and look at the next one: skip it if it's cancelled (and look again), otherwise step forward.

In [ ]:
def remove_stops(head, cancelled):
    # 1. the front: move the head past cancelled stations
    while head and head.val in cancelled:
        head = head.next

    # 2. the rest: stand on a kept station, look one ahead
    node = head
    while node and node.next:
        if node.next.val in cancelled:
            node.next = node.next.next    # skip it, stay here
        else:
            node = node.next
    return head

In [ ]:
route = ["Agra", "Delhi", "Jaipur", "Kota"]
check(to_list(remove_stops(build_list(route), {"Delhi"})), ["Agra", "Jaipur", "Kota"])
check(to_list(remove_stops(build_list(route), {"Agra"})), ["Delhi", "Jaipur", "Kota"])
check(to_list(remove_stops(build_list(route), {"Agra", "Delhi"})), ["Jaipur", "Kota"])
check(to_list(remove_stops(build_list(route), {"Delhi", "Jaipur"})), ["Agra", "Kota"])
check(to_list(remove_stops(build_list(route), {"Kota"})), ["Agra", "Delhi", "Jaipur"])
check(to_list(remove_stops(build_list(route), {"Pune"})), route)
check(remove_stops(build_list(["Agra", "Delhi"]), {"Agra", "Delhi"}), None)
check(remove_stops(None, {"Agra"}), None)

## 11. ⭐ Stretch: a game leaderboard

The dictionary answers "what is this player's best?" and the sorted list answers "how many are higher?" with one `bisect_right`. When someone improves, their old score is found with `bisect_left`, popped, and the new one goes in with `insort`, so the list stays sorted.

In [ ]:
import bisect

class Leaderboard:
    def __init__(self):
        self.best = {}       # name -> best score
        self.scores = []     # every best score, smallest first

    def submit(self, name, score):
        if name in self.best:
            old = self.best[name]
            if score <= old:
                return                          # not a new best
            self.scores.pop(bisect.bisect_left(self.scores, old))
        self.best[name] = score
        bisect.insort(self.scores, score)

    def rank(self, name):
        if name not in self.best:
            return None
        score = self.best[name]
        higher = len(self.scores) - bisect.bisect_right(self.scores, score)
        return higher + 1

    def top(self, k):
        names = sorted(self.best, key=lambda n: (-self.best[n], n))
        return names[:k]

In [ ]:
b = Leaderboard()
b.submit("Tanvi", 50)
b.submit("Ravi", 80)
b.submit("Asha", 50)
b.submit("Dev", 20)
check(b.rank("Ravi"), 1)
check(b.rank("Tanvi"), 2)
check(b.rank("Asha"), 2)
check(b.rank("Dev"), 4)
check(b.top(2), ["Ravi", "Asha"])
b.submit("Dev", 90)            # Dev's new best
check(b.rank("Dev"), 1)
check(b.rank("Ravi"), 2)
b.submit("Ravi", 10)           # lower than his best: ignored
check(b.rank("Ravi"), 2)
check(b.rank("Zara"), None)
check(b.top(10), ["Dev", "Ravi", "Asha", "Tanvi"])

## 12. ⭐ Stretch: a family tree, one generation at a time

A `deque` hands nodes back in the order they went in, so the tree comes out top level first. At the start of each round the queue holds exactly one level, so `len(queue)` says how many to take out for this inner list.

In [ ]:
from collections import deque

def levels(root):
    if root is None:
        return []
    result = []
    queue = deque([root])
    while queue:
        size = len(queue)              # this many nodes are on the current level
        level = []
        for _ in range(size):
            node = queue.popleft()
            level.append(node.val)
            if node.left:
                queue.append(node.left)
            if node.right:
                queue.append(node.right)
        result.append(level)
    return result

In [ ]:
tree = TreeNode(3, TreeNode(9), TreeNode(20, TreeNode(15), TreeNode(7)))
check(levels(tree), [[3], [9, 20], [15, 7]])
check(levels(None), [])
check(levels(TreeNode(1)), [[1]])
chain = TreeNode(1, TreeNode(2, TreeNode(3)))
check(levels(chain), [[1], [2], [3]])
full = TreeNode(1, TreeNode(2, TreeNode(4), TreeNode(5)), TreeNode(3, TreeNode(6), TreeNode(7)))
check(levels(full), [[1], [2, 3], [4, 5, 6, 7]])
odd = TreeNode(1, None, TreeNode(2, TreeNode(3), None))
check(levels(odd), [[1], [2], [3]])